<a href="https://colab.research.google.com/github/0002F16/hybrid-mobilenetv2-dualconv-eca/blob/main/colab/Train_Eval_CIFAR100_All20_MobileNetV2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/0002F16/hybrid-mobilenetv2-dualconv-eca/blob/main/colab/Train_Eval_CIFAR100_All20_MobileNetV2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Train + evaluate MobileNetV2 variants on CIFAR-100 (All 20 runs) (Colab)

This notebook trains and evaluates **4 MobileNetV2 variants** on **CIFAR-100** across 5 seeds (20 total runs):
- `baseline` (MobileNetV2 only)
- `dualconv` (DualConv B4–B10, MobileNetV2 + DualConv)
- `eca` (MobileNetV2 + ECA only)
- `hybrid` (DualConv + ECA, B4–B10)

For each run it uses `experiments/run_train_eval.py` and writes results to:
`OUTPUT_ROOT/cifar100/<model>/seed_<seed>/metrics.json`.

The CIFAR-100 training config comes from `configs/cifar100.yaml`; this notebook only overwrites `seed` per run (the runner reads `cfg["seed"]`).

Tip: set `MAX_EPOCHS` below for quick sanity runs.

In [1]:
from pathlib import Path
import os

# If you opened this notebook from a cloned repo, set REPO_DIR accordingly.
# If you need to clone, set GIT_URL to your fork/repo and run the cell.
GIT_URL = "https://github.com/0002F16/hybrid-mobilenetv2-dualconv-eca"
REPO_DIR = Path("/content/hybrid-mobilenetv2-dualconv-eca")

if not REPO_DIR.exists():
    if not GIT_URL:
        raise ValueError("Set GIT_URL to your repo URL, then re-run this cell.")
    os.system(f"git clone {GIT_URL} {REPO_DIR}")

os.chdir(REPO_DIR)
print("Repo:", REPO_DIR)

# Install pinned deps
os.system("pip -q install -r requirements.txt")

# Progress bar for the multi-run loop
os.system("pip -q install tqdm")

Repo: /content/hybrid-mobilenetv2-dualconv-eca


0

In [2]:
# Paths (feel free to change)
DATA_DIR = Path("/content/data")
OUTPUT_ROOT = Path("/content/outputs")

DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# Optional epoch cap for fast sanity runs (use None for full CIFAR-100 config: 200 epochs).
MAX_EPOCHS = None

print("DATA_DIR:", DATA_DIR)
print("OUTPUT_ROOT:", OUTPUT_ROOT)
print("MAX_EPOCHS:", MAX_EPOCHS)

DATA_DIR: /content/data
OUTPUT_ROOT: /content/outputs
MAX_EPOCHS: None


In [ ]:
import subprocess
import sys
from copy import deepcopy

import yaml
from tqdm.auto import tqdm

seeds = [42, 123, 3407, 2024, 777]
model_variants = ["baseline", "dualconv", "eca", "hybrid"]

# Controls for re-running long experiment grids.
SKIP_DONE = True
RESUME_TRAINING = True

# Generate per-seed YAML configs by copying configs/cifar100.yaml and overwriting `seed`.
base_cfg_path = Path("configs/cifar100.yaml")
base_cfg = yaml.safe_load(base_cfg_path.read_text())

GEN_CFG_DIR = Path("/content/cifar100_seed_configs")
GEN_CFG_DIR.mkdir(parents=True, exist_ok=True)

def run(cfg_path: str, *, model: str, dataset_root: Path | None = None, resume: bool = False):
    cmd = [
        sys.executable,
        "experiments/run_train_eval.py",
        "--config",
        cfg_path,
        "--output_root",
        str(OUTPUT_ROOT),
        "--model",
        model,
        "--dataset_root",
        str(dataset_root or DATA_DIR),
    ]
    if MAX_EPOCHS is not None:
        cmd += ["--max_epochs", str(int(MAX_EPOCHS))]
    if resume:
        cmd += ["--resume"]
    print(" ".join(cmd))
    subprocess.run(cmd, check=True)

def run_dir_for(seed: int, model: str) -> Path:
    # Matches `_run_dir()` in experiments/run_train_eval.py
    return OUTPUT_ROOT / "cifar100" / str(model).lower() / f"seed_{int(seed)}"

# Pre-generate all per-seed YAML configs once (cheap, keeps configs self-contained).
cfg_paths = {}
for seed in seeds:
    cfg_seed = deepcopy(base_cfg)
    cfg_seed["seed"] = int(seed)
    cfg_out_path = GEN_CFG_DIR / f"cifar100_seed_{seed}.yaml"
    cfg_out_path.write_text(yaml.safe_dump(cfg_seed, sort_keys=False), encoding="utf-8")
    cfg_paths[int(seed)] = cfg_out_path

run_specs = [(seed, model) for seed in seeds for model in model_variants]

for seed, model in tqdm(run_specs, desc="CIFAR-100 (20 runs) "):
    seed_int = int(seed)
    model_lower = str(model).lower()
    run_dir = run_dir_for(seed_int, model_lower)
    metrics_path = run_dir / "metrics.json"
    if SKIP_DONE and metrics_path.exists():
        continue

    resume = RESUME_TRAINING and (run_dir / "checkpoints" / "last.pt").exists()
    run(str(cfg_paths[seed_int]), model=model_lower, dataset_root=DATA_DIR, resume=resume)

CIFAR-100 (20 runs) :   0%|          | 0/20 [00:00<?, ?it/s]

/usr/bin/python3 experiments/run_train_eval.py --config /content/cifar100_seed_configs/cifar100_seed_42.yaml --output_root /content/outputs --model baseline --dataset_root /content/data
/usr/bin/python3 experiments/run_train_eval.py --config /content/cifar100_seed_configs/cifar100_seed_42.yaml --output_root /content/outputs --model dualconv --dataset_root /content/data


In [ ]:
# Summarize saved metrics (only CIFAR-100 runs created by this notebook's output layout)
import json

metrics_paths = sorted(OUTPUT_ROOT.glob("cifar100/*/seed_*/metrics.json"))
if not metrics_paths:
    print("No metrics.json files found under OUTPUT_ROOT/cifar100. Run the training cell first.")
else:
    for metrics_path in metrics_paths:
        data = json.loads(metrics_path.read_text())
        ds = data.get("dataset")
        model = data.get("model")
        seed = data.get("seed")
        test_top1 = data.get("test", {}).get("top1_pp")
        test_top5 = data.get("test", {}).get("top5_pp")
        best_epoch = data.get("best_val", {}).get("epoch")
        top5_str = f"{test_top5:.2f}%" if test_top5 is not None else "NA"
        print(f"{ds:12s} | {model:8s} | seed={seed} | best_epoch={best_epoch} | test_top1={test_top1:.2f}% | test_top5={top5_str} | {metrics_path}")